# 🎬 Netflix Movie Recommendation System
## Content-Based Filtering with TF-IDF & Cosine Similarity
---
### 📌 Introduction

Recommendation systems are one of the most impactful applications of machine learning in everyday life.
From Netflix to Spotify, these systems help users discover new content they are likely to enjoy.

This notebook implements a **Content-Based Filtering (CBF)** recommendation system using the **Netflix Movies Dataset**.
Instead of relying on what other users liked (collaborative filtering), CBF recommends items **similar in content** to what a user has already watched.

### 🎯 Business Questions
1. Given a movie a user has watched, **which other Netflix movies are most similar** in terms of genre, country, and description?
2. Given a set of movies a user has rated, **what movie should we recommend next** based on their content preference profile?
3. How can we **serve personalized recommendations to multiple users** simultaneously using their watch history?

### 🧠 Methodology
| Step | Technique |
|------|----------|
| Text Feature Extraction | TF-IDF Vectorizer on `description` |
| Categorical Features | Multi-hot encoding on `genre` and `country` |
| Feature Combination | Concatenate TF-IDF + genre + country vectors |
| Similarity | Cosine Similarity |
| Recommendation | Single User & Multi-User CBF |


---
## 1. 📦 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

print('Libraries loaded successfully!')

---
## 2. 📂 Load Dataset

In [ ]:
df = pd.read_csv('../netflix_titles.csv')
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

---
## 3. 🔍 Exploratory Data Analysis (EDA)

### 3.1 Missing Values

In [ ]:
# Missing value heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap')
plt.tight_layout()
plt.show()

print(df.isnull().sum())

### 3.2 Content Type Distribution (Movies vs TV Shows)

In [ ]:
type_counts = df['type'].value_counts()

plt.figure(figsize=(6, 4))
sns.barplot(x=type_counts.index, y=type_counts.values, palette='Set2')
plt.title('Content Type Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
print(type_counts)

### 3.3 Top 10 Countries by Movie Production

In [ ]:
movies_df = df[df['type'] == 'Movie'].copy()

country_counts = movies_df['country'].dropna().str.split(', ').explode().value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=country_counts.values, y=country_counts.index, palette='Blues_r')
plt.title('Top 10 Countries by Movie Production on Netflix')
plt.xlabel('Number of Movies')
plt.tight_layout()
plt.show()

### 3.4 Top 10 Genres

In [ ]:
genre_col = 'listed_in' if 'listed_in' in movies_df.columns else 'genre'
genre_counts = movies_df[genre_col].dropna().str.split(', ').explode().value_counts().head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='Reds_r')
plt.title('Top 10 Movie Genres on Netflix')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

### 3.5 Description Word Count Distribution

In [ ]:
movies_df['desc_word_count'] = movies_df['description'].dropna().apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8, 4))
sns.histplot(movies_df['desc_word_count'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Description Word Count (Movies)')
plt.xlabel('Word Count')
plt.tight_layout()
plt.show()

---
## 4. 🔧 Feature Engineering

### 4.1 Filter Movies Only & Handle Missing Values

In [ ]:
movies = df[df['type'] == 'Movie'].copy().reset_index(drop=True)

# Detect column names (Kaggle dataset may vary slightly)
genre_col = 'listed_in' if 'listed_in' in movies.columns else 'genre'

# Fill missing values
movies['description'] = movies['description'].fillna('')
movies['country'] = movies['country'].fillna('Unknown')
movies[genre_col] = movies[genre_col].fillna('Unknown')

print(f'Total movies: {len(movies)}')
movies[['title', genre_col, 'country', 'description']].head()

### 4.2 TF-IDF on Description

In [ ]:
# TF-IDF Vectorizer on description
tfidf = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_matrix = tfidf.fit_transform(movies['description'])

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Sample features: {tfidf.get_feature_names_out()[:20]}')

### 4.3 Multi-Hot Encoding for Genre & Country

In [ ]:
mlb_genre = MultiLabelBinarizer()
genre_matrix = mlb_genre.fit_transform(movies[genre_col].str.split(', '))

mlb_country = MultiLabelBinarizer()
country_matrix = mlb_country.fit_transform(movies['country'].str.split(', '))

print(f'Genre matrix shape: {genre_matrix.shape}')
print(f'Country matrix shape: {country_matrix.shape}')

### 4.4 Combine All Features into One Vector

In [ ]:
import scipy.sparse as sp

# Concatenate: TF-IDF (description) + Genre (multi-hot) + Country (multi-hot)
feature_matrix = sp.hstack([
    tfidf_matrix,
    sp.csr_matrix(genre_matrix),
    sp.csr_matrix(country_matrix)
])

print(f'Combined feature matrix shape: {feature_matrix.shape}')

---
## 5. 📐 Similarity Measure

We use **Cosine Similarity** to measure how similar two movies are based on their combined feature vectors.

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

The result ranges from **0 (no similarity)** to **1 (identical content)**.

In [ ]:
# Compute cosine similarity matrix
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)

print(f'Cosine similarity matrix shape: {cosine_sim.shape}')
print(f'Sample similarity scores (first movie vs others):\n{cosine_sim[0][:10]}')

In [ ]:
# Build a title-to-index mapping
movies = movies.reset_index(drop=True)
title_to_idx = pd.Series(movies.index, index=movies['title'].str.lower()).drop_duplicates()

print(f'Total indexed movies: {len(title_to_idx)}')

---
## 6. 🎯 Content-Based Filtering

### 6.1 Single User Recommendation

In [ ]:
def recommend_single(title, top_n=5):
    """Recommend movies similar to a given title using cosine similarity."""
    title_lower = title.lower()
    
    if title_lower not in title_to_idx:
        # Fuzzy match: show close titles
        close = [t for t in title_to_idx.index if title_lower[:4] in t]
        return f"Title not found. Did you mean: {close[:5]}?"
    
    idx = title_to_idx[title_lower]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != idx][:top_n]
    
    movie_indices = [s[0] for s in sim_scores]
    scores = [round(s[1], 4) for s in sim_scores]
    
    result = movies[['title', genre_col, 'country']].iloc[movie_indices].copy()
    result['similarity_score'] = scores
    return result.reset_index(drop=True)

In [ ]:
# Example: Recommend movies similar to 'Inception'
recommend_single('Inception', top_n=5)

In [ ]:
# Try another movie
recommend_single('The Irishman', top_n=5)

### 6.2 Multiple User Recommendation

In [ ]:
# Simulate User-Item Rating Matrix (like lecture notebook)
# 0 means not watched, non-zero = user rating (1-10)
user_item_data = {
    'user':       ['User 1', 'User 2', 'User 3', 'User 4'],
    'Inception':  [9, 8, 0, 0],
    'The Irishman':[0, 7, 9, 0],
    'Bird Box':   [8, 0, 7, 9],
    'Mank':       [0, 0, 8, 7],
}

df_user_items = pd.DataFrame(user_item_data).set_index('user')
df_user_items

In [ ]:
# Item-Feature Matrix (genre + country + description vector combined)
# We use the feature_matrix already built
# Filter feature_matrix to only user-interacted movies
watched_titles = df_user_items.columns.tolist()

# Get indices of watched movies in the movies dataframe
watched_idx = {t: title_to_idx[t.lower()] for t in watched_titles if t.lower() in title_to_idx}
print('Watched movie indices:', watched_idx)

In [ ]:
def build_user_feature_vector(user_ratings, watched_idx, feature_matrix):
    """Build a weighted user feature vector from their rated movies."""
    weighted_sum = np.zeros(feature_matrix.shape[1])
    total_rating = 0
    
    for title, rating in user_ratings.items():
        if rating > 0 and title.lower() in title_to_idx:
            idx = title_to_idx[title.lower()]
            weighted_sum += feature_matrix[idx].toarray()[0] * rating
            total_rating += rating
    
    if total_rating == 0:
        return weighted_sum
    return weighted_sum / total_rating  # Normalize

# Build user feature vectors for all users
user_feature_vectors = {}
for user in df_user_items.index:
    user_ratings = df_user_items.loc[user].to_dict()
    user_feature_vectors[user] = build_user_feature_vector(user_ratings, watched_idx, feature_matrix)

print('User feature vectors built for:', list(user_feature_vectors.keys()))

In [ ]:
def recommend_multi_user(user_name, top_n=5):
    """Recommend movies for a user based on their feature profile vector."""
    user_vec = user_feature_vectors[user_name].reshape(1, -1)
    sim_scores = cosine_similarity(user_vec, feature_matrix)[0]
    
    # Exclude already watched movies
    already_watched = [title_to_idx[t.lower()] for t in df_user_items.columns
                       if t.lower() in title_to_idx and df_user_items.loc[user_name, t] > 0]
    
    sim_scores_enum = [(i, s) for i, s in enumerate(sim_scores) if i not in already_watched]
    sim_scores_enum = sorted(sim_scores_enum, key=lambda x: x[1], reverse=True)[:top_n]
    
    indices = [s[0] for s in sim_scores_enum]
    scores = [round(s[1], 4) for s in sim_scores_enum]
    
    result = movies[['title', genre_col, 'country']].iloc[indices].copy()
    result['predicted_score'] = scores
    return result.reset_index(drop=True)

In [ ]:
# Recommend for all users
for user in df_user_items.index:
    print(f'\n🎬 Recommendations for {user}:')
    print(recommend_multi_user(user, top_n=3).to_string(index=False))

---
## 7. 📊 Summary

- **Content-Based Filtering** was implemented using TF-IDF on movie descriptions combined with multi-hot encoded genre and country features.
- **Cosine Similarity** was used to measure content similarity between movies.
- **Single User mode**: A user inputs a title they liked → top-N similar movies are returned.
- **Multi User mode**: Simulated user-rating tables are used to build a user preference profile vector, which is then scored against all unwatched movies.
- Key insight: Movies with similar **genre + country + description** vocabulary rank highest in recommendations.
